# ASMR: Agent Reasoning

This notebook demonstrates how ASMR agents reason over memories:
- RelevanceAgent: Beyond embedding similarity
- RecencyAgent: Temporal reasoning
- ConflictAgent: Contradiction detection
- SynthesisAgent: Context building

In [ ]:
import sys
sys.path.insert(0, '..')

from datetime import datetime, timedelta
from memory.schema import Memory
from agents.relevance import RelevanceAgent
from agents.recency import RecencyAgent
from agents.conflict import ConflictAgent
from agents.synthesis import SynthesisAgent

## 1. Create Test Memories

In [ ]:
now = datetime.utcnow()

# Create test memories
memories = [
    Memory(
        id="mem1",
        content="Python 3.12 introduces new performance improvements and faster startup.",
        source="python_docs",
        timestamp=now - timedelta(days=30),
        tags=["python", "performance"],
    ),
    Memory(
        id="mem2",
        content="Monty Python's Flying Circus was a British comedy series from 1969-1974.",
        source="wikipedia",
        timestamp=now - timedelta(days=100),
        tags=["comedy", "tv"],
    ),
    Memory(
        id="mem3",
        content="Python type hints help catch errors early and improve code readability.",
        source="python_tutorial",
        timestamp=now - timedelta(days=60),
        tags=["python", "typing"],
    ),
]

print(f"Created {len(memories)} test memories")

## 2. RelevanceAgent Demo

The RelevanceAgent goes beyond embedding similarity to determine actual relevance.

In [ ]:
# Using mock LLM for demo
relevance_agent = RelevanceAgent(
    llm_provider="mock",
    confidence_threshold=0.4,
)

query = "Python performance optimization"
print(f"Query: '{query}'")
print("\nIn a real scenario, the RelevanceAgent would:")
print("1. Send each memory to the LLM with the query")
print("2. Ask: 'Is this memory actually relevant to the query?'")
print("3. Filter out superficial matches (like Monty Python)")

# With mock, it returns default keep decisions
decisions = relevance_agent.reason_sync(query, memories)
for d in decisions:
    print(f"\n  Memory: {d.memory_id}")
    print(f"  Action: {d.action}")
    print(f"  Confidence: {d.confidence}")

## 3. RecencyAgent Demo

The RecencyAgent handles temporal reasoning.

In [ ]:
# Create memories with temporal aspects
policy_memories = [
    Memory(
        id="policy_old",
        content="Return policy: 30 days full refund.",
        source="policy_v1",
        timestamp=now - timedelta(days=365),
        tags=["policy"],
    ),
    Memory(
        id="policy_new",
        content="Updated: Return policy is now 15 days. Effective January 2024.",
        source="policy_v2",
        timestamp=now - timedelta(days=30),
        tags=["policy"],
    ),
]

recency_agent = RecencyAgent(
    llm_provider="mock",
    half_life_days=30,
)

query = "What is the return policy?"
print(f"Query: '{query}'")
print("\nRecencyAgent analysis:")

# Show temporal metadata
from memory.temporal import TemporalManager
tm = TemporalManager()

for mem in policy_memories:
    score = tm.recency_score(mem.timestamp)
    stale = tm.is_stale(mem)
    print(f"\n  Memory: {mem.id}")
    print(f"  Age: {(now - mem.timestamp).days} days")
    print(f"  Recency score: {score:.3f}")
    print(f"  Is stale: {stale}")

## 4. ConflictAgent Demo

In [ ]:
# Create conflicting memories
conflicting_memories = [
    Memory(
        id="ceo_old",
        content="The CEO of TechCorp is John Smith.",
        source="annual_report_2022",
        timestamp=now - timedelta(days=400),
        tags=["leadership"],
    ),
    Memory(
        id="ceo_new",
        content="Jane Doe was appointed CEO of TechCorp in March 2024.",
        source="press_release",
        timestamp=now - timedelta(days=30),
        tags=["leadership"],
    ),
]

conflict_agent = ConflictAgent(llm_provider="mock")

print("ConflictAgent would detect:")
print("\n  Conflict type: full_contradiction")
print("  Memory A: CEO is John Smith (2022)")
print("  Memory B: CEO is Jane Doe (2024)")
print("  Winner: Memory B (newer, explicitly dated)")
print("\n  Resolution: Discard old memory, keep new one")

## 5. SynthesisAgent Demo

In [ ]:
# Memories to synthesize
synthesis_memories = [
    Memory(
        id="remote_1",
        content="Employees can work remotely up to 3 days per week.",
        source="hr_policy",
        timestamp=now - timedelta(days=30),
        tags=["remote", "policy"],
    ),
    Memory(
        id="remote_2",
        content="Remote work requires manager approval.",
        source="hr_policy",
        timestamp=now - timedelta(days=30),
        tags=["remote", "policy"],
    ),
    Memory(
        id="remote_3",
        content="IT provides $500 stipend for home office setup.",
        source="it_benefits",
        timestamp=now - timedelta(days=60),
        tags=["remote", "benefits"],
    ),
]

synthesis_agent = SynthesisAgent(
    llm_provider="mock",
    max_tokens=500,
)

print("SynthesisAgent would produce:")
print("\n" + "="*50)
print("Remote Work Policy Summary")
print("="*50)
print("")
print("Employees can work remotely up to 3 days per week,")
print("subject to manager approval. [Source: hr_policy]")
print("")
print("IT provides a $500 stipend for home office equipment")
print("to support remote workers. [Source: it_benefits]")